# Notification System Design (HLD + LLD)

## 1. Problem Statement

Design a scalable notification platform capable of delivering notifications through:

- Push Notifications (FCM/APNs)
- Email
- SMS
- In-App Notifications

The system should support:
- Millions of users
- High throughput
- Retry mechanisms
- User preferences
- Scheduling
- Rate limiting
- Multi-channel delivery

---

# 2. Functional Requirements

### Core Requirements

- Send notifications to users
- Support Email, SMS, Push and In-App channels
- Template-based notifications
- Scheduled notifications
- Retry failed notifications
- User notification preferences
- Delivery tracking

### Non-Functional Requirements

- High availability
- Scalability
- Low latency
- Fault tolerance
- Observability
- Idempotency

---

# 3. Capacity Estimation

Assume:

- 50 Million Users
- 100 Million Notifications/day

Notifications/sec:

100,000,000 / 86,400 ≈ 1,157/sec

Peak Traffic:

≈ 10,000 notifications/sec

Design target:

- 50K notifications/sec

---

# 4. High Level Architecture

```

Producer Services
       |
       v
Notification API
       |
       v
Notification Service
       |
       +------------+
       |            |
       v            v
Template      Preference
Service        Service
       |
       v
Message Queue (Kafka)
       |
       v
Notification Workers
       |
+------+------+------+------+
|      |      |      |
v      v      v      v
Email SMS  Push  In-App
Gateway Gateway Gateway Gateway
```

---

# 5. Components

## Notification API

Responsibilities:

- Accept requests
- Validate payload
- Authenticate caller
- Generate notification ID

Example:

```json
{
  "userId":"123",
  "channel":"EMAIL",
  "template":"ORDER_SUCCESS"
}
```

---

## Notification Service

Responsibilities:

- Orchestration
- Preference checks
- Template rendering
- Event publishing

---

## Template Service

Stores templates:

```html
Hello {{name}},

Your order {{orderId}} has been delivered.
```

Supports:

- Email templates
- SMS templates
- Push templates

---

## User Preference Service

Stores:

```json
{
  "userId":"123",
  "emailEnabled":true,
  "smsEnabled":false,
  "pushEnabled":true
}
```

---

## Kafka

Topics:

- email_notifications
- sms_notifications
- push_notifications
- inapp_notifications

Benefits:

- Decoupling
- Reliability
- Replay capability

---

## Notification Workers

Consumes messages.

Responsibilities:

- Retry
- Channel routing
- Delivery tracking

---

# 6. Database Design

## Notification Table

| Field | Type |
|---------|---------|
| notification_id | UUID |
| user_id | String |
| channel | Enum |
| status | Enum |
| created_at | Timestamp |

### Status Values

- PENDING
- SENT
- FAILED
- RETRYING

---

## User Preferences

| Field | Type |
|---------|---------|
| user_id | String |
| email_enabled | Boolean |
| sms_enabled | Boolean |
| push_enabled | Boolean |

---

# 7. Channel Specific Design

## Email

Providers:

- SMTP
- SES
- SendGrid

Flow:

Worker → Email Gateway → SMTP Provider

---

## SMS

Providers:

- Twilio
- Sinch
- MessageBird

Flow:

Worker → SMS Provider

---

## Push

### Android

FCM

### iOS

APNs

Flow:

Worker → Push Gateway → FCM/APNs

---

## In-App

Store notifications in database.

Client pulls:

```http
GET /notifications
```

or use WebSocket.

---

# 8. Retry Strategy

Exponential Backoff

Attempt:

- 1 minute
- 5 minutes
- 15 minutes
- 1 hour

Dead Letter Queue after max retries.

---

# 9. Idempotency

Use:

```text
notification_id
```

Deduplicate before processing.

Benefits:

- Prevent duplicate emails
- Prevent duplicate SMS

---

# 10. Rate Limiting

Per User:

- 10 notifications/minute

Per Tenant:

- 10,000 notifications/minute

Implementation:

- Redis Token Bucket

---

# 11. Scheduling

Store future notifications.

Scheduler scans:

```sql
SELECT *
FROM scheduled_notifications
WHERE scheduled_time <= NOW();
```

Publishes to Kafka.

---

# 12. Observability

Metrics:

- Sent Count
- Failure Count
- Retry Count
- Latency

Tools:

- Prometheus
- Grafana
- ELK

---

# 13. Security

- Encryption at rest
- TLS
- Secret Management
- RBAC
- Audit Logs

---

# 14. Failure Handling

## Email Provider Down

Switch provider.

## Kafka Down

Persist request in DB.

## Worker Crash

Kafka replay.

---

# 15. APIs

## Send Notification

```http
POST /notifications
```

Request:

```json
{
  "userId":"123",
  "channel":"EMAIL",
  "template":"WELCOME"
}
```

---

## Get Status

```http
GET /notifications/{id}
```

---

# 16. Scalability

Scale:

- API Layer
- Kafka Consumers
- Delivery Workers

Partition Kafka by:

```text
user_id
```

Maintains ordering.

---

# 17. HLD Summary

Services:

1. Notification API
2. Notification Service
3. Template Service
4. Preference Service
5. Scheduler
6. Kafka
7. Worker Fleet
8. Email Gateway
9. SMS Gateway
10. Push Gateway
11. In-App Service

---

# 18. Interview Discussion Points

- Why Kafka?
- Retry vs DLQ
- Idempotency
- User Preferences
- Multi-Region Deployment
- Rate Limiting
- Notification Ordering
- Template Versioning
- Provider Failover
- Delivery Guarantees

This design supports large-scale notification delivery across Email, SMS, Push, and In-App channels with high availability and fault tolerance.
